# v14 — konteks beberapa menit di atas keluaran per window

Rancangan dan hipotesis dikunci di [RANCANGAN.md](RANCANGAN.md) sebelum training (sha256 di `logs/v14_praregistrasi.log`).
Tahap kedua membaca dinamika detak 2–10 menit terakhir (cara mulai, kedataran, lama keadaan cepat-teratur) plus keluaran v3.1
per window. Logika di `konteks.py`; metrik episode gaya EC57 dari `v3.3_takikardia_asli/takikardia_asli.py`.

Dijalankan dengan cwd = folder ini: `python tools/run_notebook.py v14_konteks_menit/konteks_menit.ipynb` (dari root repo).
Semua langkah berat disimpan di `../cache/v14/` dan dilewati bila sudah ada.

In [1]:
import json, sys, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
sys.path[:0] = ["../v11_model_tim", "../v3.3_takikardia_asli"]
import konteks as C
import takikardia_asli as K33
from bersama import data as D, evaluasi as E
d = D.muat()
print("window AFDB:", len(d.y), "| pasien:", len(d.patients), "| fitur konteks:", len(C.FITUR))

window AFDB: 165978 | pasien: 23 | fitur konteks: 16


## 1. OOF tahap kedua (fold resmi 5×5, bias cross-fit)

In [2]:
oof = {"xgb": C.oof_v14("xgb"), "lr": C.oof_v14("lr")}
p31, b31 = E.keputusan(d.oof_v31, d.folds, d.y)
pred, bias = {"v31": p31}, {"v31": b31}
for k, o in oof.items():
    pred[k], bias[k] = E.keputusan(o, d.folds, d.y)
tab = E.tabel_metrik(d, pred)
pd.DataFrame({k: {m: f"{v[0]:.3f} [{v[1]:.2f}–{v[2]:.2f}]" for m, v in tab[k].items() if m in ("Macro-F1","AFL F1","AFL R","AFL P","AFIB F1","Normal R")} for k in pred})

,v31,xgb,lr
Macro-F1,0.832 [0.67–0.92],0.781 [0.64–0.87],0.736 [0.65–0.82]
Normal R,0.924 [0.84–0.97],0.949 [0.88–0.99],0.946 [0.86–0.99]
AFIB F1,0.909 [0.86–0.94],0.922 [0.87–0.96],0.921 [0.86–0.96]
AFL P,0.715 [0.26–0.95],0.498 [0.07–0.83],0.438 [0.17–0.58]
AFL R,0.592 [0.13–0.91],0.477 [0.17–0.75],0.282 [0.07–0.63]
AFL F1,0.647 [0.17–0.89],0.473 [0.05–0.73],0.338 [0.11–0.57]


In [3]:
bd = {k: E.banding(d, pred[k], p31) for k in oof}
pd.DataFrame({k: {m: f"Δ {v[0]:+.3f} [{v[1]:+.3f}…{v[2]:+.3f}], P = {v[3]:.3f} → {E.vonis(v[3])}" for m, v in b.items()} for k, b in bd.items()})

,xgb,lr
Macro-F1,"Δ -0.051 [-0.121…+0.005], P = 0.033 → lebih buruk","Δ -0.095 [-0.137…+0.076], P = 0.066 → tidak te..."
AFL F1,"Δ -0.174 [-0.381…-0.014], P = 0.022 → lebih buruk","Δ -0.309 [-0.425…+0.195], P = 0.060 → tidak te..."
AFL P,"Δ -0.217 [-0.434…+0.085], P = 0.117 → tidak te...","Δ -0.277 [-0.420…+0.002], P = 0.026 → lebih buruk"
AFL R,"Δ -0.114 [-0.287…+0.052], P = 0.041 → lebih buruk","Δ -0.309 [-0.423…+0.319], P = 0.087 → tidak te..."
Normal R,"Δ +0.025 [+0.009…+0.042], P = 1.000 → lebih baik","Δ +0.022 [+0.008…+0.034], P = 0.998 → lebih baik"


In [4]:
afl = {k: E.afl_per_pasien(d, pred[k]) for k in pred}
palsu = {k: E.afl_palsu(d, pred[k]) for k in pred}
display(pd.DataFrame(afl).round(2).rename_axis("recall AFL per pasien"))
display(pd.DataFrame(palsu).round(0).rename_axis("AFL palsu (window/ulangan)"))
for k in pred:
    print(k, "bias [N, AFIB, AFL] median", np.median(bias[k].reshape(-1, 3), 0), "| fraksi di batas grid", E.bias_di_batas(bias[k]))

,v31,xgb,lr
recall AFL per pasien,,,
04043,0.40,0.00,0.20
04908,0.92,0.21,0.80
04936,0.51,0.39,0.07
06426,0.17,0.09,0.07
07910,0.00,0.21,0.50
08215,0.00,0.00,0.04
08378,0.96,0.83,0.63


,v31,xgb,lr
AFL palsu (window/ulangan),,,
total,274.0,936.0,463.0
dari_normal,55.0,710.0,166.0
dari_afib,219.0,227.0,297.0
pasien_tanpa_afl,25.0,56.0,109.0


v31 bias [N, AFIB, AFL] median [ 0.   -0.75 -4.5 ] | fraksi di batas grid 0.0
xgb bias [N, AFIB, AFL] median [ 0.  -0.5 -2.5] | fraksi di batas grid 0.06
lr bias [N, AFIB, AFL] median [ 0.   -0.5  -0.25] | fraksi di batas grid 0.0


## 2. Metrik episode (EC57: sensitivitas dan +P episode/durasi), rata-rata 5 ulangan

In [5]:
ep = {}
for k in pred:
    for kelas, n in ((1, "AFIB"), (2, "AFL")):
        ms = [K33.metrik_episode(d.records, d.t, d.y, pred[k][r], kelas) for r in range(5)]
        ep[f"{k} {n}"] = {m: float(np.mean([x[m] for x in ms])) for m in ms[0]}
pd.DataFrame(ep).T.round(3)

,se_episode,ppv_episode,se_durasi,ppv_durasi,n_ref,n_det,palsu_per_jam
v31 AFIB,0.970,0.562,0.932,0.887,881.0,6252.4,11.858
v31 AFL,0.686,0.403,0.592,0.715,14.0,198.2,0.521
xgb AFIB,0.918,0.706,0.917,0.928,881.0,2481.4,3.123
xgb AFL,0.443,0.333,0.477,0.498,14.0,228.8,0.763
lr AFIB,0.873,0.752,0.929,0.913,881.0,2677.2,2.870
lr AFL,0.486,0.125,0.282,0.438,14.0,126.6,0.490


## 3. H3: sinus takikardia asli (EPHNOGRAM, window langkah 5 s, detektor perangkat) dan H4: uji stres konteks sintetis

In [6]:
p3, p4 = C.CACHE / "h3_ephnogram.npz", C.CACHE / "h4_sintetis.json"
if not (p3.exists() and p4.exists()):
    m14, b14 = C.model_penuh("xgb")
    h4 = C.h4_sintetis(m14, b14); json.dump(h4, open(p4, "w"), indent=1)
    np.savez_compressed(p3, **C.h3_ephnogram(m14, b14))
H = dict(np.load(p3)); h4 = json.load(open(p4))
m = H["hr"] >= 120
h3 = {}
for k in ("k31", "k14"):
    t, ci, _ = K33._boot_subjek(H["subjek"][m], (H[k][m] != 0).astype(float))
    h3[k] = dict(alarm=t, ci=ci.tolist())
h3["n"], h3["subjek"] = int(m.sum()), int(np.unique(H["subjek"][m]).size)
pita = {}
for lo, hi in ((100, 110), (110, 120), (120, 140), (140, 999)):
    mm = (H["hr"] >= lo) & (H["hr"] < hi)
    pita[f"{lo}–{hi if hi < 999 else '∞'} bpm"] = dict(n=int(mm.sum()), v31=float((H["k31"][mm] != 0).mean()), v14=float((H["k14"][mm] != 0).mean()))
print(json.dumps(h3, indent=1)); display(pd.DataFrame(pita).T.round(3)); display(pd.DataFrame(h4).T.round(3))

{
 "k31": {
  "alarm": 0.9827586206896551,
  "ci": [
   0.9634085660563079,
   0.9942229909886096
  ]
 },
 "k14": {
  "alarm": 0.9576572008113591,
  "ci": [
   0.9238714874164803,
   0.9814218612443891
  ]
 },
 "n": 3944,
 "subjek": 17
}


,n,v31,v14
100–110 bpm,3713.0,0.155,0.100
110–120 bpm,2515.0,0.628,0.430
120–140 bpm,2464.0,0.975,0.933
140–∞ bpm,1480.0,0.996,0.999


,v31,v14
sinus_120_sd10,1.0,0.993
sinus_120_sd20,1.0,0.594
sinus_140_sd10,1.0,1.000
sinus_140_sd20,1.0,1.000
sinus_160_sd10,1.0,1.000
sinus_160_sd20,1.0,1.000
afl_150_sd10,1.0,0.992
afl_150_sd20,1.0,0.992


## 4. Vonis hipotesis (kriteria RANCANGAN bagian 4)

In [7]:
dx = bd["xgb"]["Macro-F1"]
vonis = {
 "H1 tidak lebih buruk (P > 0,05)": bool(dx[3] > 0.05),
 "H2 Se episode AFL turun ≤ 0,10 dan AFL palsu/jam turun": bool(ep["v31 AFL"]["se_episode"] - ep["xgb AFL"]["se_episode"] <= 0.10 and ep["xgb AFL"]["palsu_per_jam"] < ep["v31 AFL"]["palsu_per_jam"]),
 "H3 alarm EPHNOGRAM ≥120 bpm turun ≥ separuh": bool(h3["k14"]["alarm"] <= 0.5 * h3["k31"]["alarm"]),
 "H4 alarm sinus datar turun ≥ separuh & deteksi AFL ≥ 0,8× v3.1": bool(np.mean([v["v14"] for k, v in h4.items() if k.startswith("sinus")]) <= 0.5 * np.mean([v["v31"] for k, v in h4.items() if k.startswith("sinus")]) and all(v["v14"] >= 0.8 * v["v31"] for k, v in h4.items() if k.startswith("afl"))),
}
vonis

{'H1 tidak lebih buruk (P > 0,05)': False,
 'H2 Se episode AFL turun ≤ 0,10 dan AFL palsu/jam turun': False,
 'H3 alarm EPHNOGRAM ≥120 bpm turun ≥ separuh': False,
 'H4 alarm sinus datar turun ≥ separuh & deteksi AFL ≥ 0,8× v3.1': False}

In [8]:
ring = {"metrik": tab, "banding_v31": bd, "afl_per_pasien": afl, "afl_palsu": palsu,
        "bias_median": {k: np.median(b.reshape(-1, 3), 0).tolist() for k, b in bias.items()},
        "bias_di_batas": {k: E.bias_di_batas(b) for k, b in bias.items()}, "episode": ep, "h3": h3, "h3_pita": pita, "h4": h4,
        "vonis": vonis, "lingkungan": E._lingkungan()}
(C.CACHE / "ringkasan.json").write_text(json.dumps(ring, indent=1, default=float, ensure_ascii=False))
print("tersimpan:", C.CACHE / "ringkasan.json")

tersimpan: /mnt/HDD/03_Akademik_dan_Tugas/WrapSihedaf/cache/v14/ringkasan.json


## 5. Kesimpulan

Dijalankan di PC (RTX 5060, CPU 12 core) lewat `tools/run_notebook.py`, 6 November 2026. **Keempat hipotesis yang dikunci tidak lolos; v14 bukan kandidat.**

- **H1, AFDB:** XGBoost Macro-F1 0,781 [0,64–0,87] vs v3.1 0,832: Δ −0,051, P = 0,033 → **lebih buruk**. AFL F1 −0,174 (P = 0,022). Regresi logistik 0,736 (P = 0,066, tidak terbedakan, tetapi presisi AFL lebih buruk).
- **H2, episode:** Se episode AFL 0,686 → 0,443 (turun 0,24, batas 0,10). AFL palsu 274 → 936 window per ulangan (dari ritme Normal 55 → 710); episode AFL palsu 0,52 → 0,76 per jam.
- **Satu perbaikan:** episode AFIB palsu 11,9 → 3,1 per jam (+P episode 0,56 → 0,71). Konteks membantu AF yang terpecah-pecah.
- **H3, EPHNOGRAM (sinus takikardia asli, 17 subjek, 3.944 window ≥ 120 bpm):** alarm 98,3% [96–99] → 95,8% [92–98]. Di 110–120 bpm 62,8% → 43,0%.
- **H4, uji stres konteks:** sinus yang naik bertahap tetap dialarmkan ±100% (kecuali 120 bpm jitter 20 ms: 59%); AFL 2:1 mendadak tetap terdeteksi 99%.
- **Recall AFL per pasien (XGBoost):** 04043 0,00 · 04908 0,21 · 04936 0,39 · 06426 0,09 · 07910 0,21 · 08215 0,00 · 08378 0,83 (v3.1: 0,40 · 0,92 · 0,51 · 0,17 · 0,00 · 0,00 · 0,96).
- **Kenapa:** AFDB tidak punya episode sinus takikardia, jadi tahap kedua tidak punya contoh "naik bertahap = sinus"; dengan 7 pasien AFL, fitur konteks menghafal pola per pasien.
- **Yang tidak bisa disimpulkan:** apakah konteks berguna bila data latih memuat episode sinus takikardia asli. EPHNOGRAM dipakai sebagai uji, jadi tidak boleh sekaligus dipakai melatih tanpa rancangan baru (subjek terpisah).